# Mix Paxos2020 videos into the Paxos2025 training set

Draws random Paxos2020 videos (grade shown), weighting grades by the inverse of their count in the current mixed
training set (starts as the Paxos2025 `mil_train.json`), so underrepresented grades come up more often. Accept or reject
each one; decisions are saved after every click (resumable), and the mixed set is written to `OUT_JSON` in the split dir.

Train on it with `--split_path <SPLIT_DIR> --train_json mil_train_mixed.json --fused_dataset` (no `--cluster_sample`:
there are no Paxos2020 cluster files).

In [1]:
import io, json, os, random
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import ipywidgets as w
from IPython.display import display
from PIL import Image

if Path.cwd().name == "data":  # opened from data/ -> work from the repo root like the training scripts
    os.chdir(Path.cwd().parent)

In [2]:
SPLIT_DIR = Path("data/stratified_splits_paxos2025/split_1")
BASE_JSON = SPLIT_DIR / "mil_train.json"            # Paxos2025 training set
POOL_JSON = Path("data/clip_labels_2020.json")      # Paxos2020 candidates (video -> grade)
VIDEO_DIR = Path("data/ensemble_results_paxos2020/cleaned_videos")
OUT_JSON = SPLIT_DIR / "mil_train_mixed.json"
DECISIONS_JSON = SPLIT_DIR / "mil_train_mixed_decisions.json"  # {video: "add" | "reject"}, for resuming
TARGET = None         # optional {grade: max count in mixed set}, e.g. {0: 89, 1: 50, 3: 50}; grades at target aren't drawn
N_PREVIEW = 12        # frames shown per video
THUMB = 220           # px per thumbnail
GIF_SIZE = 256        # px of the "Play clip" animation (built on demand, ~4 s per clip)
GIF_STEP = 4          # use every n-th frame for the animation
SEED = 42
GRADES = range(5)

In [3]:
base = json.load(open(BASE_JSON))
pool = {k: v for k, v in json.load(open(POOL_JSON)).items() if k not in base}
decisions = json.load(open(DECISIONS_JSON)) if DECISIONS_JSON.exists() else {}
rng = random.Random(SEED + len(decisions))
frame_rng = np.random.default_rng()  # separate, so re-rolling previews doesn't change the video draws

def find_video(name):
    stem = name.rsplit(".", 1)[0]
    hits = [p for p in VIDEO_DIR.glob(f"*{stem}.*") if p.stem in (stem, f"CLEAN_{stem}")]
    return hits[0] if hits else None

missing = [k for k in pool if find_video(k) is None]
pool = {k: v for k, v in pool.items() if k not in missing}
print(f"base: {len(base)} videos | pool: {len(pool)} Paxos2020 videos ({len(missing)} without cleaned clip) | "
      f"already decided: {len(decisions)}")

def mixed():
    return {**base, **{k: pool[k] for k, d in decisions.items() if d == "add" and k in pool}}

def counts(d):
    c = Counter(d.values())
    return [c.get(g, 0) for g in GRADES]

def undecided():
    return [k for k in pool if k not in decisions]

def grade_probs():
    """Draw probability per grade: 1 / (count in mixed set), over grades with undecided videos left and below TARGET."""
    left, cur = counts({k: pool[k] for k in undecided()}), counts(mixed())
    wts = {g: 1 / max(cur[g], 1) for g in GRADES
           if left[g] and (TARGET is None or cur[g] < TARGET.get(g, float("inf")))}
    total = sum(wts.values())
    return {g: wt / total for g, wt in wts.items()}

def draw():
    """Pick a grade by grade_probs(), then a random undecided video of that grade."""
    probs = grade_probs()
    if not probs:
        return None
    g = rng.choices(list(probs), weights=list(probs.values()))[0]
    return rng.choice([k for k in undecided() if pool[k] == g])

def save():
    with open(DECISIONS_JSON, "w") as f:
        json.dump(decisions, f, indent=4)
    with open(OUT_JSON, "w") as f:
        json.dump(mixed(), f, indent=4)

def preview(path, shuffle=False):
    """N_PREVIEW evenly spaced (with shuffle: random, sorted) frames as one PNG grid, plus frame count and duration in
    seconds (None if fps unknown)."""
    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    thumbs = []
    idx = (np.sort(frame_rng.choice(n, min(N_PREVIEW, n), replace=False)) if shuffle
           else np.linspace(0, n - 1, N_PREVIEW).astype(int))
    for i in idx:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        thumbs.append(cv2.resize(frame, (THUMB, THUMB)) if ok else np.zeros((THUMB, THUMB, 3), np.uint8))
    cap.release()
    cols = 6
    thumbs += [np.zeros_like(thumbs[0])] * (-len(thumbs) % cols)
    grid = np.vstack([np.hstack(thumbs[r:r + cols]) for r in range(0, len(thumbs), cols)])
    return cv2.imencode(".png", grid)[1].tobytes(), n, (n / fps if fps > 0 else None)

def clip_gif(path):
    """Downscaled animated GIF of the clip (browsers can't play the FMP4-encoded cleaned clips directly)."""
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frames, i = [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % GIF_STEP == 0:
            frames.append(Image.fromarray(cv2.cvtColor(cv2.resize(frame, (GIF_SIZE, GIF_SIZE)), cv2.COLOR_BGR2RGB)).quantize(64))
        i += 1
    cap.release()
    buf = io.BytesIO()
    frames[0].save(buf, format="GIF", save_all=True, append_images=frames[1:], duration=1000 * GIF_STEP / fps, loop=0)
    return buf.getvalue()

base: 236 videos | pool: 341 Paxos2020 videos (0 without cleaned clip) | already decided: 91


In [4]:
title = w.HTML()
image = w.Image(format="png")
clip = w.Image(format="gif", layout=w.Layout(display="none"))
stats = w.HTML()
btn_add = w.Button(description="Add", button_style="success", icon="plus")
btn_rej = w.Button(description="Reject", button_style="danger", icon="times")
btn_undo = w.Button(description="Undo last", icon="undo")
btn_play = w.ToggleButton(description="Play clip", icon="play")
btn_frames = w.Button(description="Other frames", icon="refresh")
added_sel = w.SelectMultiple(rows=8, description="Added", layout=w.Layout(width="420px"))
btn_remove = w.Button(description="Remove selected", button_style="warning", icon="minus")
history, current = [], [None]

def render_stats():
    b, m, p = counts(base), counts(mixed()), counts({k: pool[k] for k in undecided()})
    probs = grade_probs()
    rows = "".join(f"<tr><td>{g}</td><td>{b[g]}</td><td><b>{m[g]}</b></td><td>{m[g] - b[g]:+d}</td><td>{p[g]}</td>"
                   f"<td>{probs.get(g, 0):.0%}</td></tr>" for g in GRADES)
    stats.value = ("<table style='text-align:right'><tr><th>grade</th><th>2025 train</th><th>mixed</th>"
                   f"<th>added</th><th>pool left</th><th>P(draw)</th></tr>{rows}<tr><td>sum</td><td>{sum(b)}</td>"
                   f"<td><b>{sum(m)}</b></td><td>{sum(m) - sum(b):+d}</td><td>{sum(p)}</td><td></td></tr></table>")
    added_sel.options = sorted(k for k, d in decisions.items() if d == "add")

def render_clip(*_):
    if btn_play.value and current[0] is not None:
        clip.value = clip_gif(find_video(current[0]))
        clip.layout.display = None
    else:
        clip.value = b""
        clip.layout.display = "none"

def show_next():
    current[0] = draw()
    render_stats()
    if current[0] is None:
        title.value = "<h3>No undecided Paxos2020 videos left (or all grades at TARGET).</h3>"
        image.value = b""
        render_clip()
        return
    png, n, dur = preview(find_video(current[0]))
    image.value = png
    dur_str = f"{dur:.1f} s" if dur is not None else "? s"
    title.value = f"<h3>{current[0]} &mdash; grade {pool[current[0]]} &mdash; {n} frames &mdash; {dur_str}</h3>"
    render_clip()

def reshuffle(_):
    if current[0] is not None:
        image.value = preview(find_video(current[0]), shuffle=True)[0]

def decide(choice):
    if current[0] is None:
        return
    decisions[current[0]] = choice
    history.append(current[0])
    save()
    show_next()

def undo(_):
    if history:
        decisions.pop(history.pop(), None)
        save()
        show_next()

def remove(_):
    for k in added_sel.value:
        decisions[k] = "reject"
    save()
    render_stats()

btn_add.on_click(lambda _: decide("add"))
btn_rej.on_click(lambda _: decide("reject"))
btn_undo.on_click(undo)
btn_play.observe(render_clip, names="value")
btn_frames.on_click(reshuffle)
btn_remove.on_click(remove)

display(w.VBox([title, w.HBox([image, clip], layout=w.Layout(gap="20px")),
                w.HBox([btn_add, btn_rej, btn_undo, btn_frames, btn_play]),
                w.HBox([stats, w.VBox([added_sel, btn_remove])], layout=w.Layout(gap="40px"))]))
show_next()

Summary of the saved mixed set:

In [6]:
m = json.load(open(OUT_JSON)) if OUT_JSON.exists() else mixed()
print(f"{OUT_JSON}: {len(m)} videos, grades {counts(m)} (2025 train: {counts(base)})")

data/stratified_splits_paxos2025/split_1/mil_train_mixed.json: 289 videos, grades [94, 33, 78, 34, 50] (2025 train: [89, 17, 53, 28, 49])


Write the mixed set for every split in `SPLIT_DIR.parent` (same picks, each split's own Paxos2025 train set), and copy the decisions so the notebook can resume there. Re-run after changing picks.

In [7]:
added = {k: pool[k] for k, d in decisions.items() if d == "add" and k in pool}
for d in sorted(p.parent for p in SPLIT_DIR.parent.glob("split_*/mil_train.json")):
    if d != SPLIT_DIR and json.load(open(d / "mil_train.json")).keys() & added.keys():
        print(f"{d.name}: skipped, its mil_train.json already contains picked videos")
        continue
    split_mixed = {**json.load(open(d / "mil_train.json")), **added}
    with open(d / OUT_JSON.name, "w") as f:
        json.dump(split_mixed, f, indent=4)
    with open(d / DECISIONS_JSON.name, "w") as f:
        json.dump(decisions, f, indent=4)
    print(f"{d / OUT_JSON.name}: {len(split_mixed)} videos, grades {counts(split_mixed)}")

data/stratified_splits_paxos2025/split_1/mil_train_mixed.json: 289 videos, grades [94, 33, 78, 34, 50]
data/stratified_splits_paxos2025/split_2/mil_train_mixed.json: 290 videos, grades [94, 33, 78, 35, 50]
data/stratified_splits_paxos2025/split_3/mil_train_mixed.json: 290 videos, grades [93, 34, 78, 35, 50]
data/stratified_splits_paxos2025/split_4/mil_train_mixed.json: 290 videos, grades [94, 34, 77, 35, 50]
data/stratified_splits_paxos2025/split_5/mil_train_mixed.json: 290 videos, grades [94, 34, 78, 35, 49]


NotADirectoryError: [Errno 20] Not a directory: 'data/stratified_splits_paxos2025/split_metadata.csv/mil_train.json'